# Dataset inspection

This notebook inspects the retail transaction data for:
- data types
- missing values
- unusual quantities and prices
- cancelled transactions

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

repo_root = Path.cwd()
if not (repo_root / "data" / "raw" / "Online Retail.xlsx").exists():
    repo_root = repo_root.parent

DATA_PATH = repo_root / "data" / "raw" / "Online Retail.xlsx"
print(f"Loading dataset from: {DATA_PATH}")

excel_file = pd.ExcelFile(DATA_PATH)
print("Available sheets:", excel_file.sheet_names)

df = pd.read_excel(DATA_PATH, sheet_name=excel_file.sheet_names[0])
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# 1) Data types
print("Data types:\n")
print(df.dtypes)

print("\nPreview of columns:")
print(list(df.columns))

In [ ]:
# 2) Missing values
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("Missing values summary:\n")
print(missing_summary)

In [ ]:
# 3) Unusual quantities and prices
quantity_series = pd.to_numeric(df["Quantity"], errors="coerce")
price_series = pd.to_numeric(df["UnitPrice"], errors="coerce")

print("Quantity summary:\n")
print(quantity_series.describe())

print("\nUnit price summary:\n")
print(price_series.describe())

# IQR-based outlier detection
q1_qty, q3_qty = quantity_series.quantile([0.25, 0.75])
iqr_qty = q3_qty - q1_qty
lower_qty = q1_qty - 1.5 * iqr_qty
upper_qty = q3_qty + 1.5 * iqr_qty

q1_price, q3_price = price_series.quantile([0.25, 0.75])
iqr_price = q3_price - q1_price
lower_price = q1_price - 1.5 * iqr_price
upper_price = q3_price + 1.5 * iqr_price

unusual_quantity = df[(quantity_series < lower_qty) | (quantity_series > upper_qty)]
unusual_price = df[(price_series < lower_price) | (price_series > upper_price)]

print(f"\nUnusual quantity rows: {len(unusual_quantity)}")
print(unusual_quantity[["InvoiceNo", "StockCode", "Description", "Quantity"]].head(10))

print(f"\nUnusual price rows: {len(unusual_price)}")
print(unusual_price[["InvoiceNo", "StockCode", "Description", "UnitPrice"]].head(10))

In [ ]:
# 4) Cancelled transactions
# In retail data, cancelled invoices are often prefixed with 'C'
df["InvoiceNo_str"] = df["InvoiceNo"].astype(str)

cancelled_transactions = df[df["InvoiceNo_str"].str.startswith("C", na=False)].copy()

print(f"Cancelled transaction count: {len(cancelled_transactions)}")
print(cancelled_transactions[["InvoiceNo", "Quantity", "UnitPrice", "CustomerID"]].head(10))